# Chatbot using Hugging Face Transformers

## Objective
To build a conversational chatbot using a pre-trained transformer model (DialoGPT) that generates meaningful responses.

## Tools Used
- Python
- Hugging Face Transformers
- PyTorch
- Google Colab

## Model Used
- microsoft/DialoGPT-medium

## Features
- Interactive chatbot
- Context-aware conversation
- Dynamic response generation
- Exit condition (exit/quit)

## Step 1: Install Required Libraries

In [ ]:
# Install required libraries
!pip install transformers
!pip install torch

## Step 2: Load Pre-trained Transformer Model (DialoGPT)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load pre-trained model and tokenizer
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model and Tokenizer loaded successfully!")

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model and Tokenizer loaded successfully!


## Step 3: Generate Response for User Input

In [ ]:
# Encode user input and generate response

user_input = "Hello"

# Convert text to tensor (numbers)
input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

# Generate response
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id
)

# Decode response
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)

print("Chatbot:", response)

Chatbot: Hello! :D


## Step 4: Build Continuous Conversation Chatbot

In [ ]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

chat_history_ids = None  # stores conversation history

while True:
    user_input = input("You: ")

    # Exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Encode input with EOS token
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append to chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.75
    )

    # Decode only the new response
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

You: helolo, how are you
Chatbot: im fine
You: what is java
Chatbot: sorry, i forgot a word. i'm not sure how to use java
You: exit
Chatbot: Goodbye! 👋


## Step 5: Improve Response Quality and Performance

In [ ]:
print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

chat_history_ids = None

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! 👋")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Limit conversation history (VERY IMPORTANT)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids[:, -1000:], new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response (improved settings)
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=300,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=30,
        top_p=0.85,
        temperature=0.6,
        no_repeat_ngram_size=3,
        repetition_penalty=1.2
    )

    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    # Basic filtering
    if len(response.strip()) == 0:
        response = "I didn't quite understand that. Could you rephrase?"

    # Optional: fix awkward tone
    if "sleepy" in response.lower():
        response = "I'm not sure, but you seem fine! How can I help you?"

    print("Chatbot:", response)

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

You: hello, how are you?
Chatbot: I'm good, thanks. How about yourself?
You: I am good. What is java?
Chatbot: It's a programming language.
You: What is a chatbot?
Chatbot: A program that allows people to communicate in real time via voice and text.
You: exit
Chatbot: Goodbye! 👋
